In [0]:
%run ../00_Utils/utils

## Venda do Varejo - Prata
- O objetivo aqui é termos uma tabela com IDs unicos por venda, agrupando o que for necessario, como quantidades e valores. Deixando uma tabela com registros unicos por venda, facilita a analise posteriormente. 
- Nessa tabela, os produtos não são mencionados, somente a quantidade de produtos, para isso teremos outra tabela prata especifica para os produtos de cada venda. 

In [0]:
df_vendas = spark.table("workspace.lakehouse_panvel.bronze_vendas")

In [0]:

df_vendas_silver = (
    df_vendas
    .groupBy(
        "id_venda",
        "id_filial",
        "data_venda",
        "id_cliente_fidelidade",
        "forma_pagamento",
        "status_venda",
        "valor_total_venda"
    )
    .agg(
        F.count("*").alias("quantidade_produtos"),
        F.sum("quantidade").alias("quantidade_itens_produtos"),
        F.sum("valor_bruto_item").alias("valor_bruto_venda"),
        F.sum("valor_desconto_item").alias("valor_desconto_venda")
    )
    .withColumn(
        "flag_fidelidade",
        F.when(
            F.col("id_cliente_fidelidade").isNotNull() & (F.trim(F.col("id_cliente_fidelidade")) != ""),
            F.lit("Sim")
        ).otherwise(F.lit("Não"))
    )
    .select(
        "id_venda",
        "id_filial",
        "data_venda",
        "quantidade_produtos",
        "quantidade_itens_produtos",
        "valor_bruto_venda",
        "valor_desconto_venda",
        "valor_total_venda",
        "id_cliente_fidelidade",
        "flag_fidelidade",
        "forma_pagamento",
        "status_venda"
    )
    .orderBy("id_venda")
)


In [0]:
df_vendas_silver.write.mode("overwrite").saveAsTable("workspace.lakehouse_panvel.silver_vendas")

## Vendas do Varejo (Produtos) -  Prata

In [0]:
df_itens_venda = (
    df_vendas
    .select(
        "id_venda",
        "id_item_venda",
        "id_filial",
        "id_produto",
        "quantidade",
        "valor_unitario",
        "valor_bruto_item",
        "valor_desconto_item"
    )
    .withColumn(
        "valor_final_item",
        F.round(
            F.col("valor_bruto_item") - F.col("valor_desconto_item"),
            2
        )
    )
    .orderBy("id_venda", "id_item_venda")
)

In [0]:
df_itens_venda.write.mode("overwrite").saveAsTable("workspace.lakehouse_panvel.silver_itens_venda")

### Produtos e Categoirias
- "Qual venda de produtos da categoria própria em relação a rede?"
- Por essa pergunta deduzo que categoria "Panvel" = "Propria" e outras = "Rede", como isso definido, criei uma nova coluna já dentro do dataframe Produtos.
- Fazendo uso direto na tabela de produtos sem precisar consultar a tabela categoria. 

In [0]:
df_produtos = spark.table("workspace.lakehouse_panvel.bronze_produtos")
df_categoria = spark.table("workspace.lakehouse_panvel.bronze_categorias")

In [0]:
df_produtos_silver = (
    df_produtos
    .join(df_categoria, "id_categoria", "left")
    .withColumn(
        "nome_categoria",
        F.when(
            F.col("nome_categoria").isNull(),
            F.lit("Sem Categoria")
        ).otherwise(F.col("nome_categoria"))
    )
    .withColumn(
        "tipo_categoria",
        F.when(
            F.col("nome_categoria") == "Panvel",
            F.lit("Própria")
        ).otherwise(F.lit("Rede"))
    )
)

In [0]:
df_produtos_silver.write.mode("overwrite").saveAsTable("workspace.lakehouse_panvel.silver_produtos")